# 0b. Audit Output Tambahan

Melengkapi Fase 1: bikin `00_file_inventory.csv`, `01_schema_comparison.csv`, `02_data_quality.csv`, plus cek **duplikat penuh**, **distribusi kode**, **nilai ekstrem**, dan **konsentrasi penghapusan per tanggal** (buat Fase 2A).

Semua output ditulis ke `out/output_eda/`.

# 0. Import

In [ ]:
import os
import pandas as pd
import numpy as np

# 1. Hitung Per Bulan

In [ ]:
RAW = "../datasets/raw/2026"
valid_zone = set(range(1, 264))
valid_vendor = {1, 2, 6, 7}
valid_rate = {1, 2, 3, 4, 5, 6, 99}
valid_pay = {0, 1, 2, 3, 4, 5, 6}

inventory, quality, schemas = [], [], {}
konsentrasi = {}


def catat_tanggal(rule, dates):
    dd = konsentrasi.setdefault(rule, {})
    for k, v in pd.Series(dates).dt.date.value_counts().items():
        dd[k] = dd.get(k, 0) + int(v)


for m in range(1, 7):
    path = f"{RAW}/yellow_tripdata_2026-{m:02d}.parquet"
    d = pd.read_parquet(path)
    n = len(d)
    schemas[m] = dict(zip(d.columns, d.dtypes.astype(str)))

    inventory.append({
        "bulan": m, "file": os.path.basename(path),
        "ukuran_mb": round(os.path.getsize(path) / 1024 ** 2, 2),
        "jumlah_baris": n, "jumlah_kolom": d.shape[1],
        "pickup_min": d["tpep_pickup_datetime"].min(), "pickup_max": d["tpep_pickup_datetime"].max(),
        "dropoff_min": d["tpep_dropoff_datetime"].min(), "dropoff_max": d["tpep_dropoff_datetime"].max(),
    })

    dur = (d["tpep_dropoff_datetime"] - d["tpep_pickup_datetime"]).dt.total_seconds() / 60
    quality.append({
        "bulan": m, "jumlah_baris": n, "duplikat_penuh": int(d.duplicated().sum()),
        "pickup_di_luar_rentang": int(((d["tpep_pickup_datetime"] < "2026-01-01") | (d["tpep_pickup_datetime"] > "2026-06-30 23:59:59")).sum()),
        "dropoff_di_luar_rentang": int(((d["tpep_dropoff_datetime"] < "2026-01-01") | (d["tpep_dropoff_datetime"] > "2026-06-30 23:59:59")).sum()),
        "durasi_invalid": int(((dur <= 0) | (dur > 1440)).sum()),
        "dist_neg": int((d["trip_distance"] < 0).sum()),
        "dist_nol": int((d["trip_distance"] == 0).sum()),
        "dist_gt100": int((d["trip_distance"] > 100).sum()),
        "fare_neg": int((d["fare_amount"] < 0).sum()),
        "fare_gt500": int((d["fare_amount"] > 500).sum()),
        "total_neg": int((d["total_amount"] < 0).sum()),
        "total_gt500": int((d["total_amount"] > 500).sum()),
        "pass_nan": int(d["passenger_count"].isna().sum()),
        "pass_nol": int((d["passenger_count"] == 0).sum()),
        "pass_neg": int((d["passenger_count"] < 0).sum()),
        "pass_gt6": int((d["passenger_count"] > 6).sum()),
        "vendor_unknown": int((~d["VendorID"].isin(valid_vendor)).sum()),
        "ratecode_unknown": int((d["RatecodeID"].notna() & ~d["RatecodeID"].isin(valid_rate)).sum()),
        "payment_unknown": int((~d["payment_type"].isin(valid_pay)).sum()),
        "pu_unknown": int(((d["PULocationID"].isna()) | (~d["PULocationID"].isin(valid_zone))).sum()),
        "do_unknown": int(((d["DOLocationID"].isna()) | (~d["DOLocationID"].isin(valid_zone))).sum()),
    })

    # konsentrasi penghapusan per tanggal (urutan sama seperti cleaning)
    x = d.copy()
    m1 = (x["tpep_pickup_datetime"] < "2026-01-01") | (x["tpep_pickup_datetime"] > "2026-06-30 23:59:59")
    catat_tanggal("C1", x.loc[m1, "tpep_pickup_datetime"]); x = x[~m1]
    m2 = x["tpep_pickup_datetime"].isna() | x["tpep_dropoff_datetime"].isna()
    catat_tanggal("C2", x.loc[m2, "tpep_pickup_datetime"]); x = x[~m2]
    dur2 = (x["tpep_dropoff_datetime"] - x["tpep_pickup_datetime"]).dt.total_seconds() / 60
    m3 = (dur2 <= 0) | (dur2 > 1440)
    catat_tanggal("C3", x.loc[m3, "tpep_pickup_datetime"]); x = x[~m3]
    m4 = x["PULocationID"].isna() | (~x["PULocationID"].isin(valid_zone))
    catat_tanggal("C4", x.loc[m4, "tpep_pickup_datetime"]); x = x[~m4]
    m5 = x["DOLocationID"].isna() | (~x["DOLocationID"].isin(valid_zone))
    catat_tanggal("C5", x.loc[m5, "tpep_pickup_datetime"]); x = x[~m5]
    m6 = x.duplicated(keep="first")
    catat_tanggal("C6", x.loc[m6, "tpep_pickup_datetime"]); x = x[~m6]
    m7 = (x["trip_distance"] < 0) | (x["trip_distance"] > 100)
    catat_tanggal("C7", x.loc[m7, "tpep_pickup_datetime"]); x = x[~m7]
    m8 = (x["fare_amount"] < 0) | (x["fare_amount"] > 500) | (x["total_amount"] < 0) | (x["total_amount"] > 500)
    catat_tanggal("C8", x.loc[m8, "tpep_pickup_datetime"]); x = x[~m8]
    m9 = (x["passenger_count"] < 0) | (x["passenger_count"] > 6)
    catat_tanggal("C9", x.loc[m9, "tpep_pickup_datetime"]); x = x[~m9]
    m10 = (~x["VendorID"].isin(valid_vendor)) | (~x["payment_type"].isin(valid_pay)) | (x["RatecodeID"].notna() & ~x["RatecodeID"].isin(valid_rate))
    catat_tanggal("C10", x.loc[m10, "tpep_pickup_datetime"]); x = x[~m10]

    print(f"Bulan {m} selesai: {n:,} baris")

del d, x

# 2. Simpan 00 / 01 / 02

In [ ]:
os.makedirs("../out/output_eda", exist_ok=True)

inventory_df = pd.DataFrame(inventory)
inventory_df.loc[len(inventory_df)] = {
    "bulan": "Total", "file": "", "ukuran_mb": round(inventory_df["ukuran_mb"].sum(), 2),
    "jumlah_baris": int(inventory_df["jumlah_baris"].sum()), "jumlah_kolom": "",
    "pickup_min": "", "pickup_max": "", "dropoff_min": "", "dropoff_max": ""}
inventory_df.to_csv("../out/output_eda/00_file_inventory.csv", index=False)
display(inventory_df)

# schema comparison
semua_kolom = []
for m in schemas:
    for c in schemas[m]:
        if c not in semua_kolom:
            semua_kolom.append(c)
baris = []
for c in semua_kolom:
    dtypes = {m: schemas[m].get(c, "-") for m in range(1, 7)}
    konsisten = len({v for v in dtypes.values() if v != "-"}) == 1
    baris.append({"kolom": c, **{f"bulan_{m}": dtypes[m] for m in range(1, 7)}, "konsisten": konsisten})
schema_df = pd.DataFrame(baris)
schema_df.to_csv("../out/output_eda/01_schema_comparison.csv", index=False)
display(schema_df)

quality_df = pd.DataFrame(quality)
quality_df.to_csv("../out/output_eda/02_data_quality.csv", index=False)
display(quality_df)

# 3. Konsentrasi Penghapusan per Tanggal

In [ ]:
rows = []
for rule, dd in konsentrasi.items():
    total = sum(dd.values())
    if total == 0:
        rows.append({"aturan": rule, "total_dibuang": 0, "tanggal_paling_banyak": "-", "jumlah_di_tanggal_itu": 0, "%_di_tanggal_itu": 0.0})
        continue
    tgl, jml = max(dd.items(), key=lambda kv: kv[1])
    rows.append({"aturan": rule, "total_dibuang": total, "tanggal_paling_banyak": str(tgl),
                 "jumlah_di_tanggal_itu": jml, "%_di_tanggal_itu": round(jml / total * 100, 2)})

konsentrasi_df = pd.DataFrame(rows)
konsentrasi_df.to_csv("../out/output_eda/04_cleaning_concentration.csv", index=False)
display(konsentrasi_df)

# 4. Cek Koordinat di Wilayah NYC

In [ ]:
zona = pd.read_csv("../datasets/zone/zone_centroid_lookup.csv")
# bounding box kasar NYC
lat_min, lat_max = 40.4, 41.0
lon_min, lon_max = -74.3, -73.6
luar = zona[(zona["latitude"] < lat_min) | (zona["latitude"] > lat_max) |
            (zona["longitude"] < lon_min) | (zona["longitude"] > lon_max)]
print("Total zona:", len(zona))
print("Zona di luar bounding box NYC:", len(luar))
print("Latitude range:", zona["latitude"].min(), "-", zona["latitude"].max())
print("Longitude range:", zona["longitude"].min(), "-", zona["longitude"].max())
if len(luar):
    display(luar[["LocationID", "zone_name", "borough", "latitude", "longitude"]])

## Catatan

- `duplikat_penuh` dihitung pakai `DataFrame.duplicated()` (semua kolom).
- Nilai ekstrem pakai batas kasar (jarak > 100 mil, tarif > $500) yang sama dengan aturan cleaning.
- `%_di_tanggal_itu` = porsi baris yang dibuang pada satu tanggal dengan penghapusan terbanyak. Kalau kecil, penghapusan tidak terkonsentrasi di satu tanggal.